In [ ]:
from pathlib import Path
import pandas as pd
from music21 import converter, expressions, chord, note
import music21


"""
Parse unfolded BPSD MusicXML scores into note-level tables and augment them with:
1. motif labels from motif-note CSV annotations
2. chord annotations derived from cleaned BPSD chord files

Data sources:
- BPSD v2 unfolded score XML files
- motif-note CSV annotations
- cleaned BPSD chord annotations

Important parts of the parsing / structural extraction workflow are based on the BPSD
processing approach associated with Zeitler et al. The present version was
adapted for the motif-transformation analysis pipeline used in this repository.

Notes:
- This pipeline currently targets first-movement files under the repository's
  established naming convention.
- The motif alignment uses a staged matching procedure:
    1. strict onset tolerance + exact pitch + exact staff
    2. strict onset tolerance + pitch tolerance of 1 semitone + exact staff
    3. wider onset tolerance + exact pitch + exact staff
"""

# ---------------------------------------------------------------------
# Repository root
# ---------------------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# ---------------------------------------------------------------------
# Sonata name mapping
# ---------------------------------------------------------------------
name_mapping = {
    "01": "Beethoven_Op002No1",
    "02": "Beethoven_Op002No2",
    "03": "Beethoven_Op002No3",
    "04": "Beethoven_Op007",
    "05": "Beethoven_Op010No1",
    "06": "Beethoven_Op010No2",
    "07": "Beethoven_Op010No3",
    "08": "Beethoven_Op013",
    "09": "Beethoven_Op014No1",
    "10": "Beethoven_Op014No2",
    "11": "Beethoven_Op022",
    "12": "Beethoven_Op026",
    "13": "Beethoven_Op027No1",
    "14": "Beethoven_Op027No2",
    "15": "Beethoven_Op028",
    "16": "Beethoven_Op031No1",
    "17": "Beethoven_Op031No2",
    "18": "Beethoven_Op031No3",
    "19": "Beethoven_Op049No1",
    "20": "Beethoven_Op049No2",
    "21": "Beethoven_Op053",
    "22": "Beethoven_Op054",
    "23": "Beethoven_Op057",
    "24": "Beethoven_Op078",
    "25": "Beethoven_Op079",
    "26": "Beethoven_Op081a",
    "27": "Beethoven_Op090",
    "28": "Beethoven_Op101",
    "29": "Beethoven_Op106",
    "30": "Beethoven_Op109",
    "31": "Beethoven_Op110",
    "32": "Beethoven_Op111"
}

# ---------------------------------------------------------------------
# Base paths
# ---------------------------------------------------------------------
xml_folder = ROOT / "data" / "raw" / "BPSD_v2" / "0_RawData" / "score_xml_unfolded"
output_folder = ROOT / "data" / "processed" / "parsed_features"
csv_label_folder = ROOT / "data" / "raw" / "Beethoven_motif" / "csv_notes"
csv_chord_data_folder = ROOT / "data" / "processed" / "cleaned_data" / "ann_score_chord_clean"

output_folder.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Process all mapped sonatas
# ---------------------------------------------------------------------
# Loop over the mapping dictionary
for dictionary_key, dictionary_value in name_mapping.items():
    # Paths
    file_path = xml_folder / f"{dictionary_value}-01.xml"
    output_path = output_folder / f"extracted_{dictionary_key}-1.csv"
    csv_label_with_staff_path = csv_label_folder / f"{dictionary_key}-1.csv"
    csv_chord_data_path = csv_chord_data_folder / f"{dictionary_value}-01_clean.csv"

    if not file_path.exists():
        print(f"Skipping {dictionary_key}: missing XML file {file_path.name}")
        continue
    if not csv_label_with_staff_path.exists():
        print(f"Skipping {dictionary_key}: missing motif label file {csv_label_with_staff_path.name}")
        continue
    if not csv_chord_data_path.exists():
        print(f"Skipping {dictionary_key}: missing chord annotation file {csv_chord_data_path.name}")
        continue

    # Load data
    df_motifs = pd.read_csv(csv_label_with_staff_path)
    df_chords = pd.read_csv(csv_chord_data_path)

    print(f"Processing {file_path.name}")

    s = converter.parse(file_path)
    s = s.stripTies()

    df = pd.DataFrame(columns=[
        "Type",
        "start_meas",
        "end_meas",
        "duration_quarterLength",
        "pitch",
        "pitchName",
        "timeSig",
        "articulation",
        "grace",
        "Clef",
        "Onset",
        "Offset",
        "Is_Chord",
        "Fermata",
        "Ornaments",
        "Slurs"
    ])

    # Extract notes and rests
    for part_index, part in enumerate(s.parts):
        clef = 0 if part_index == 0 else 1  # Clef: 0 for right hand, 1 for left hand

        for element in part.flat.notesAndRests:
            measure = element.measureNumber
            timeSig = element.getContextByClass("TimeSignature")
            timeSigRatio = timeSig.numerator / timeSig.denominator if timeSig else 1
            beatDur = element.beatDuration.quarterLength / 4
            noteDur = element.duration.quarterLength / 4
            beat = element.beat - 1
            beatNr = beat * beatDur

            articulation = "_".join([art.name for art in element.articulations]) if hasattr(element, "articulations") else None

            # Compute Onset and Offset
            onset = element.offset
            offset = onset + element.quarterLength

            # Check for Fermatas
            fermata_present = False
            if hasattr(element, "expressions") and element.expressions:
                fermata_present = any(isinstance(expr, expressions.Fermata) for expr in element.expressions)
            elif hasattr(element, "notations") and element.notations:
                fermata_present = any(hasattr(notation, "fermata") for notation in element.notations)

            if isinstance(element, note.Note) or isinstance(element, chord.Chord):
                pitches = element.pitches if isinstance(element, chord.Chord) else [element.pitch]
                for pitch in pitches:
                    df = pd.concat([df, pd.DataFrame.from_dict({
                        "Type": ["Note"],
                        "start_meas": [measure + beatNr / timeSigRatio],
                        "end_meas": [measure + (beatNr + noteDur) / timeSigRatio],
                        "duration_quarterLength": [max(element.quarterLength, 1 / 16)],
                        "pitch": [pitch.midi],
                        "pitchName": [pitch.nameWithOctave],
                        "timeSig": ["%i/%i" % (timeSig.numerator, timeSig.denominator) if timeSig else "N/A"],
                        "articulation": [articulation],
                        "grace": [int(element.duration.isGrace)],
                        "Clef": [clef],
                        "Onset": [onset],
                        "Offset": [offset],
                        "Is_Chord": [1 if isinstance(element, chord.Chord) else 0],
                        "Fermata": [int(fermata_present)],
                        "Ornaments": [None if not element.expressions else ", ".join([ornament.classes[0] for ornament in element.expressions if isinstance(ornament, expressions.Ornament)])],
                        "Slurs": [1 if hasattr(element, "getSpannerSites") and any(isinstance(sp, music21.spanner.Slur) for sp in element.getSpannerSites()) else 0]
                    })])
            elif element.isRest:
                df = pd.concat([df, pd.DataFrame.from_dict({
                    "Type": ["Rest"],
                    "start_meas": [measure + beatNr / timeSigRatio],
                    "end_meas": [measure + (beatNr + noteDur) / timeSigRatio],
                    "duration_quarterLength": [max(element.quarterLength, 1 / 16)],
                    "pitch": [None],
                    "pitchName": [None],
                    "timeSig": ["%i/%i" % (timeSig.numerator, timeSig.denominator) if timeSig else "N/A"],
                    "articulation": [None],
                    "grace": [0],
                    "Clef": [clef],
                    "Onset": [onset],
                    "Offset": [offset],
                    "Is_Chord": [0],
                    "Fermata": [int(fermata_present)],
                    "Ornaments": [None],
                    "Slurs": [0]
                })])

    # Sort by start_meas and Clef
    df.sort_values(by=["start_meas", "Clef"], inplace=True)

    # Adjust for Auftakt (pickup measure adjustment)
    if (df["start_meas"].min() - 1.0) > 0:
        print("Adjusting for Auftakt: subtracting 1 measure.")
        df[["start_meas", "end_meas"]] = df[["start_meas", "end_meas"]] - 1.0
        df["Onset"] = df["Onset"] - 1.0
        df["Offset"] = df["Offset"] - 1.0

    # Make "Type" the first column
    df = df[["Type"] + [col for col in df.columns if col != "Type"]]

    # reset index
    df.reset_index(drop=True, inplace=True)

    # Create Motif column
    df["Motif"] = None

    # Add motif labels based on onset (with tolerance) and pitch
    unmatched_motifs = []  # To track unmatched motifs

    # First pass: match based on onset tolerance (0.05) and pitch
    for _, motif_row in df_motifs.iterrows():
        motif_onset = motif_row["onset"]
        motif_pitch = motif_row["midi_number"]
        motif_type = motif_row["type"]
        motif_staff = motif_row["staff_number"]

        # Ignore rows where motif_type is NaN
        if pd.isna(motif_type):
            continue

        # Check if the motif exists in the parsed DataFrame
        matched_rows = df[
            (df["Type"] == "Note") &
            (abs(df["Onset"] - motif_onset) <= 0.05) &
            (df["pitch"] == motif_pitch) &
            (df["Clef"] == motif_staff)
        ]

        if matched_rows.empty:
            # Store unmatched motif for further processing
            unmatched_motifs.append({
                "onset": motif_onset,
                "pitch": motif_pitch,
                "type": motif_type,
                "staff": motif_staff
            })
        else:
            # Assign motif to matched rows
            df.loc[
                (df["Type"] == "Note") &
                (abs(df["Onset"] - motif_onset) <= 0.05) &
                (df["pitch"] == motif_pitch) &
                (df["Clef"] == motif_staff),
                "Motif"
            ] = motif_type

    # Second pass: assign unmatched motifs based on staff and pitch tolerance
    for unmatched in unmatched_motifs:
        motif_onset = unmatched["onset"]
        motif_pitch = unmatched["pitch"]
        motif_type = unmatched["type"]
        motif_staff = unmatched["staff"]

        # Find notes in the same staff with pitch tolerance of 1
        candidate_rows = df[
            (df["Type"] == "Note") &
            (abs(df["Onset"] - motif_onset) <= 0.05) &
            (abs(df["pitch"] - motif_pitch) <= 1) &
            (df["Clef"] == motif_staff) &  # Ensure same staff
            (df["Motif"].isna())  # Only consider notes without a motif assigned
        ]

        if not candidate_rows.empty:
            # Assign motif to these notes
            df.loc[candidate_rows.index, "Motif"] = motif_type

    # Third pass: assign unmatched motifs with higher onset tolerance (0.5) and no pitch tolerance
    for unmatched in unmatched_motifs:
        motif_onset = unmatched["onset"]
        motif_pitch = unmatched["pitch"]
        motif_type = unmatched["type"]
        motif_staff = unmatched["staff"]

        # Find notes within the higher onset tolerance and same staff
        candidate_rows = df[
            (df["Type"] == "Note") &
            (abs(df["Onset"] - motif_onset) <= 0.5) &
            (df["pitch"] == motif_pitch) &
            (df["Clef"] == motif_staff) &  # Ensure same staff
            (df["Motif"].isna())  # Only consider notes without a motif assigned
        ]

        if not candidate_rows.empty:
            # Assign motif to these notes
            df.loc[candidate_rows.index, "Motif"] = motif_type

    # Add chord features to notes
    chord_features = [
        "shorthand", "extended", "majminInv", "majmin",
        "majminNumeric", "quality", "inversion", "roman", "localkey"
    ]

    for feature in chord_features:
        df[feature] = None  # Initialize the columns for chord features

    # Match and assign chord features
    for _, chord_row in df_chords.iterrows():
        start_meas = chord_row["start"]
        end_meas = chord_row["end"]

        for feature in chord_features:
            df.loc[
                (df["Type"] == "Note") &
                (df["start_meas"] >= start_meas) &
                (df["end_meas"] <= end_meas),
                feature
            ] = chord_row[feature]

    # Reset index and save
    df.reset_index(drop=True, inplace=True)
    df.to_csv(output_path, index=False)
    print(f"Parsed data saved to {output_path.name}")